# Improvement iteration 3: an LLM program generator fine-tuned with LoRA (GPU)

**Question.** Does a larger, instruction-tuned decoder-only model (Qwen2.5-1.5B-Instruct, about 6x the
parameters of `t5-base`) write better FinQA programs from the same inputs?

**What changes.** Only the generator. Inputs (question plus top-5 retrieved facts, training with gold
plus random distractors resampled each epoch), targets (the published `program`), the executor and the
grounded beam selection of Notebook 14 are unchanged. The base model is frozen; LoRA adapters (rank 16)
are trained on every linear layer. Settings: `configs/generator_qwen_lora.toml`.

**Decision this iteration tests.** Whether to replace the Notebook 11 generator. The bar is the current
pipeline: 547 dev records correct (0.6195) with grounded selection.

**Protocol.** Train on train; after each epoch decode dev greedily and keep the adapter with the best dev
execution accuracy; then decode the 4 beam candidates for the final report. Test is not loaded. Needs a
CUDA GPU (Kaggle T4); `FINEVIDENCE_SMOKE_TEST=1` runs a tiny CPU check.

In [ ]:
from __future__ import annotations

import json
import os
import platform
import random
import subprocess
import time
import tomllib
from pathlib import Path

import numpy as np
import pandas as pd
import peft
import torch
import transformers
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup

from finevidence.data import load_finqa_split
from finevidence.data.acquisition import download_finqa_splits
from finevidence.generation import (
    build_retrieved_examples,
    build_training_examples,
    load_rankings,
    score_program_prediction,
    select_grounded_candidate,
    summarize_scores,
    verify_rankings_file,
)
from finevidence.generation.causal import (
    IGNORE_INDEX,
    INSTRUCTION,
    PROGRAM_MARKER,
    build_causal_example,
    extract_program,
)

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
SMOKE_TEST = os.environ.get("FINEVIDENCE_SMOKE_TEST") == "1"
cfg = tomllib.loads((PROJECT_ROOT / "configs" / "generator_qwen_lora.toml").read_text())
assert "test" not in (cfg["train_split"], cfg["evaluation_split"])
pretrained = (
    "trl-internal-testing/tiny-Qwen2ForCausalLM-2.5"
    if SMOKE_TEST
    else cfg["model"]["pretrained_name"]
)
epochs = 1 if SMOKE_TEST else cfg["training"]["epochs"]
num_beams = 2 if SMOKE_TEST else cfg["generation"]["num_beams"]
suffix = "-smoke" if SMOKE_TEST else ""
model_dir = Path(
    os.environ.get("FINEVIDENCE_MODEL_DIR") or PROJECT_ROOT / cfg["output"]["model_dir"]
)
model_dir = Path(str(model_dir) + suffix)
predictions_dir = PROJECT_ROOT / cfg["output"]["predictions_dir"]
rankings_path = PROJECT_ROOT / cfg["inputs"]["rankings_path"]

seed = cfg["seed"]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not SMOKE_TEST and device.type != "cuda":
    raise RuntimeError(
        "Full training needs a CUDA GPU; set FINEVIDENCE_SMOKE_TEST=1 for a CPU check."
    )
use_fp16 = device.type == "cuda"
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=PROJECT_ROOT
).stdout.strip()
environment = {
    "commit": commit,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
    "smoke_test": SMOKE_TEST,
    "pretrained": pretrained,
    "epochs": epochs,
    "num_beams": num_beams,
    "fp16": use_fp16,
    "seed": seed,
}
print(json.dumps(environment, indent=2))

## 1. Verified data, rankings and inputs

In [ ]:
raw_dir = PROJECT_ROOT / cfg["dataset_root"]
print(download_finqa_splits(raw_dir, (cfg["train_split"], cfg["evaluation_split"])))
verify_rankings_file(rankings_path, cfg["inputs"]["rankings_sha256"])
rankings = load_rankings(rankings_path)
train = load_finqa_split(raw_dir / f"{cfg['train_split']}.json")
dev = load_finqa_split(raw_dir / f"{cfg['evaluation_split']}.json")
if SMOKE_TEST:
    train, dev = train[:24], dev[:8]
facts = cfg["training"]["facts_per_input"]
retrieved_examples = build_retrieved_examples(dev, rankings, facts_per_input=facts)
print({"train_records": len(train), "dev_records": len(dev), "test_loaded": False})

## 2. Tokenization

The prompt is the instruction, the source and the marker `program:`; only the program and the
end-of-sequence token are trained (`finevidence.generation.causal`).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(pretrained)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
max_source_tokens = cfg["model"]["max_source_tokens"]
marker_ids = tokenizer(PROGRAM_MARKER, add_special_tokens=False).input_ids


def prompt_ids(example) -> list[int]:
    source = tokenizer(INSTRUCTION + example.source, add_special_tokens=False).input_ids
    return [*source[:max_source_tokens], *marker_ids]


def training_features(example) -> tuple[list[int], list[int]]:
    source = tokenizer(INSTRUCTION + example.source, add_special_tokens=False).input_ids
    target = tokenizer(" " + example.target, add_special_tokens=False).input_ids
    return build_causal_example(
        source, marker_ids, target, tokenizer.eos_token_id, max_source_tokens=max_source_tokens
    )


first = build_training_examples(train, facts_per_input=facts, seed=seed)
lengths = np.array([len(training_features(example)[0]) for example in first])
truncated = sum(
    len(tokenizer(INSTRUCTION + e.source, add_special_tokens=False).input_ids) > max_source_tokens
    for e in first
)
print(
    f"train sequence tokens: median {int(np.median(lengths))}, p95 {int(np.percentile(lengths, 95))}, "
    f"max {lengths.max()}; sources truncated: {truncated}/{len(first)}"
)

## 3. Train LoRA adapters

The base weights stay frozen in fp16; the adapters are kept in fp32 and trained under fp16 autocast with
loss scaling. Each optimizer step averages `micro_batch_size x gradient_accumulation` examples over their
target tokens. After every epoch, dev is decoded greedily and the best adapter is saved.

In [ ]:
def clock(seconds: float) -> str:
    minutes, seconds = divmod(int(seconds), 60)
    hours, minutes = divmod(minutes, 60)
    return f"{hours}h{minutes:02d}m{seconds:02d}s" if hours else f"{minutes}m{seconds:02d}s"


base = AutoModelForCausalLM.from_pretrained(
    pretrained, torch_dtype=torch.float16 if use_fp16 else torch.float32
).to(device)
base.gradient_checkpointing_enable()
base.enable_input_require_grads()
base.config.use_cache = False
lora = cfg["lora"]
model = get_peft_model(
    base,
    LoraConfig(
        r=lora["rank"],
        lora_alpha=lora["alpha"],
        lora_dropout=lora["dropout"],
        target_modules="all-linear",
        task_type="CAUSAL_LM",
    ),
)
for parameter in model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()
model.print_trainable_parameters()


def pad_right(rows: list[list[int]], value: int) -> torch.Tensor:
    width = max(len(row) for row in rows)
    return torch.tensor([row + [value] * (width - len(row)) for row in rows], device=device)


@torch.no_grad()
def decode(examples, beams: int) -> list[list[str]]:
    model.eval()
    model.config.use_cache = True
    tokenizer.padding_side = "left"
    outputs = []
    for start in range(0, len(examples), cfg["generation"]["eval_batch_size"]):
        batch = examples[start : start + cfg["generation"]["eval_batch_size"]]
        prompts = [prompt_ids(example) for example in batch]
        width = max(len(p) for p in prompts)
        input_ids = torch.tensor(
            [[tokenizer.pad_token_id] * (width - len(p)) + p for p in prompts], device=device
        )
        attention = torch.tensor(
            [[0] * (width - len(p)) + [1] * len(p) for p in prompts], device=device
        )
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_fp16):
            generated = model.generate(
                input_ids=input_ids,
                attention_mask=attention,
                max_new_tokens=cfg["model"]["max_new_tokens"],
                num_beams=beams,
                num_return_sequences=beams,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        texts = tokenizer.batch_decode(generated[:, width:], skip_special_tokens=True)
        programs = [extract_program(text) for text in texts]
        outputs.extend(programs[i : i + beams] for i in range(0, len(programs), beams))
    model.config.use_cache = False
    return outputs


def score(examples, programs) -> tuple[list, dict]:
    scores = [
        score_program_prediction(record, program, example.fact_texts)
        for record, example, program in zip(dev, examples, programs, strict=True)
    ]
    return scores, summarize_scores(scores)


training = cfg["training"]
step_size = training["micro_batch_size"] * training["gradient_accumulation"]
steps_per_epoch = -(-len(train) // step_size)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=training["learning_rate"]
)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(training["warmup_ratio"] * steps_per_epoch * epochs), steps_per_epoch * epochs
)
scaler = torch.amp.GradScaler(enabled=use_fp16)
history, best = [], -1
model_dir.mkdir(parents=True, exist_ok=True)
run_started = time.time()
report_every = max(1, steps_per_epoch // 10)
for epoch in range(epochs):
    model.train()
    examples = build_training_examples(train, facts_per_input=facts, seed=seed + epoch)
    random.Random(seed * 1_000_003 + epoch).shuffle(examples)
    features = [training_features(example) for example in examples]
    started, losses = time.time(), []
    print(f"\nEpoch {epoch + 1}/{epochs}", flush=True)
    for step, start in enumerate(range(0, len(features), step_size), start=1):
        batch = features[start : start + step_size]
        tokens = sum(sum(label != IGNORE_INDEX for label in labels) for _, labels in batch)
        optimizer.zero_grad(set_to_none=True)
        step_loss = 0.0
        for chunk_start in range(0, len(batch), training["micro_batch_size"]):
            chunk = batch[chunk_start : chunk_start + training["micro_batch_size"]]
            input_ids = pad_right([ids for ids, _ in chunk], tokenizer.pad_token_id)
            labels = pad_right([labels for _, labels in chunk], IGNORE_INDEX)
            attention = pad_right([[1] * len(ids) for ids, _ in chunk], 0)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_fp16):
                logits = model(input_ids=input_ids, attention_mask=attention).logits
            shift_logits = logits[:, :-1].float()
            shift_labels = labels[:, 1:]
            loss = (
                torch.nn.functional.cross_entropy(
                    shift_logits.reshape(-1, shift_logits.size(-1)),
                    shift_labels.reshape(-1),
                    ignore_index=IGNORE_INDEX,
                    reduction="sum",
                )
                / tokens
            )
            scaler.scale(loss).backward()
            step_loss += loss.item()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        losses.append(step_loss)
        if step % report_every == 0 or step == steps_per_epoch:
            elapsed = time.time() - started
            print(
                f"  step {step}/{steps_per_epoch} | loss {np.mean(losses[-report_every:]):.4f} | "
                f"elapsed {clock(elapsed)} | epoch left ~{clock(elapsed / step * (steps_per_epoch - step))}",
                flush=True,
            )
    train_seconds = time.time() - started
    greedy = [candidates[0] for candidates in decode(retrieved_examples, beams=1)]
    _, summary = score(retrieved_examples, greedy)
    history.append(
        {
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "train_seconds": round(train_seconds, 1),
            "epoch_seconds_with_dev": round(time.time() - started, 1),
            **{f"dev_greedy_{k}": v for k, v in summary.items() if k != "records"},
        }
    )
    if summary["execution_correct"] > best:
        best = summary["execution_correct"]
        model.save_pretrained(model_dir / "best")
        tokenizer.save_pretrained(model_dir / "best")
    if device.type == "cuda":
        history[-1]["peak_gpu_gb"] = round(torch.cuda.max_memory_allocated() / 1024**3, 2)
    (model_dir / "history.json").write_text(
        json.dumps({"environment": environment, "history": history}, indent=2)
    )
    print(
        f"Epoch {epoch + 1} done: {history[-1]} | total {clock(time.time() - run_started)}",
        flush=True,
    )
display(pd.DataFrame(history))

## 4. Best adapter: top beam and grounded selection

The best adapter decodes 4 beam candidates per dev record; `top_beam` takes the first and `grounded`
takes the first that executes and uses only numbers in the facts.

In [ ]:
model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(
        pretrained, torch_dtype=torch.float16 if use_fp16 else torch.float32
    ),
    model_dir / "best",
).to(device)
model.config.use_cache = False
started = time.time()
candidates = decode(retrieved_examples, beams=num_beams)
print(f"decoded {len(candidates)} records with {num_beams} beams in {clock(time.time() - started)}")
results = {}
for selection in ("top_beam", "grounded"):
    chosen = [
        options[
            0
            if selection == "top_beam"
            else select_grounded_candidate(options, example.fact_texts, record["table"])
        ]
        for record, example, options in zip(dev, retrieved_examples, candidates, strict=True)
    ]
    scores, results[f"retrieved_{selection}"] = score(retrieved_examples, chosen)
    path = (
        predictions_dir
        / f"{cfg['experiment_name']}{suffix}_dev_retrieved_{selection}_predictions.jsonl"
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for example, record_score in zip(retrieved_examples, scores, strict=True):
            handle.write(
                json.dumps({**vars(record_score), "fact_keys": example.fact_keys, "commit": commit})
                + "\n"
            )
path = predictions_dir / f"{cfg['experiment_name']}{suffix}_dev_candidates.jsonl"
with path.open("w", encoding="utf-8") as handle:
    for example, options in zip(retrieved_examples, candidates, strict=True):
        handle.write(
            json.dumps({"record_id": example.record_id, "candidates": options, "commit": commit})
            + "\n"
        )
display(pd.DataFrame(results).T)